In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [9]:
df = pd.read_csv("Time_Wasters_on_Social_Media.csv")
print(df.head())
print(df.columns)

   UserID  Age  Gender       Location  Income   Debt  Owns Property  \
0       1   56    Male       Pakistan   82812   True           True   
1       2   46  Female         Mexico   27999  False           True   
2       3   32  Female  United States   42436  False           True   
3       4   60    Male         Barzil   62963   True          False   
4       5   25    Male       Pakistan   22096  False           True   

      Profession Demographics   Platform  ...  ProductivityLoss  Satisfaction  \
0       Engineer        Rural  Instagram  ...                 3             7   
1         Artist        Urban  Instagram  ...                 5             5   
2       Engineer        Rural   Facebook  ...                 6             4   
3  Waiting staff        Rural    YouTube  ...                 3             7   
4        Manager        Urban     TikTok  ...                 8             2   

      Watch Reason  DeviceType       OS  Watch Time  Self Control  \
0  Procrastinatio

In [11]:
## Data Cleaning
df.columns = (
    df.columns
    .str.strip()
    .str.replace(" ", "_", regex=False)
)

print(df.columns)

Index(['UserID', 'Age', 'Gender', 'Location', 'Income', 'Debt',
       'Owns_Property', 'Profession', 'Demographics', 'Platform',
       'Total_Time_Spent', 'Number_of_Sessions', 'Video_ID', 'Video_Category',
       'Video_Length', 'Engagement', 'Importance_Score', 'Time_Spent_On_Video',
       'Number_of_Videos_Watched', 'Scroll_Rate', 'Frequency',
       'ProductivityLoss', 'Satisfaction', 'Watch_Reason', 'DeviceType', 'OS',
       'Watch_Time', 'Self_Control', 'Addiction_Level', 'CurrentActivity',
       'ConnectionType'],
      dtype='object')


In [13]:
selected_cols = [
    "Total_Time_Spent",
    "Number_of_Sessions",
    "Number_of_Videos_Watched",
    "Scroll_Rate",
    "Frequency",
    "Engagement",
    "Video_Category",
    "Watch_Reason",
    "Self_Control",
    "Satisfaction",
    "Addiction_Level",
    "ProductivityLoss"
]

profile_df = df[selected_cols].copy()
print(profile_df.head())

   Total_Time_Spent  Number_of_Sessions  Number_of_Videos_Watched  \
0                80                  17                        22   
1               228                  14                        31   
2                30                   6                         7   
3               101                  19                        41   
4               136                   6                        21   

   Scroll_Rate  Frequency  Engagement Video_Category     Watch_Reason  \
0           87      Night        7867         Pranks  Procrastination   
1           46  Afternoon        5944         Pranks            Habit   
2           88    Evening        8674          Vlogs    Entertainment   
3           93      Night        2477          Vlogs            Habit   
4            4    Morning        3093         Gaming          Boredom   

   Self_Control  Satisfaction  Addiction_Level  ProductivityLoss  
0             5             7                5                 3  
1           

In [14]:
# check missing
print(profile_df.isna().sum())

Total_Time_Spent            0
Number_of_Sessions          0
Number_of_Videos_Watched    0
Scroll_Rate                 0
Frequency                   0
Engagement                  0
Video_Category              0
Watch_Reason                0
Self_Control                0
Satisfaction                0
Addiction_Level             0
ProductivityLoss            0
dtype: int64


In [20]:
numeric_cols = [
    "Total_Time_Spent",
    "Number_of_Sessions",
    "Number_of_Videos_Watched",
    "Scroll_Rate",
    "Engagement",
    "Self_Control",
    "Satisfaction"
]

In [21]:
categorical_cols = [
    "Video_Category",
    "Watch_Reason"
]

In [22]:
# Standardize
scaler = StandardScaler()
profile_df[numeric_cols] = scaler.fit_transform(profile_df[numeric_cols])

In [23]:
# check result after standardization
print(profile_df[numeric_cols].head())

   Total_Time_Spent  Number_of_Sessions  Number_of_Videos_Watched  \
0         -0.850977            1.299273                 -0.231634   
1          0.912804            0.741406                  0.410208   
2         -1.446849           -0.746240                 -1.301370   
3         -0.600710            1.671184                  1.123366   
4         -0.183600           -0.746240                 -0.302949   

   Scroll_Rate  Engagement  Self_Control  Satisfaction  
0     1.275597    0.986675     -1.017757      1.006976  
1    -0.129321    0.325532     -0.045687      0.064115  
2     1.309863    1.264128      0.440348     -0.407316  
3     1.481195   -0.866451     -1.017757      1.006976  
4    -1.568505   -0.654666      1.412417     -1.350177  


In [24]:
# usage_score
profile_df["usage_score"] = profile_df[
    ["Total_Time_Spent", "Number_of_Sessions", "Number_of_Videos_Watched"]
].mean(axis=1)

In [26]:
# interaction_score
profile_df["interaction_score"] = profile_df[
    ["Scroll_Rate", "Engagement"]
].mean(axis=1)

In [27]:
# self_reg_score & self_reg_risk
profile_df["self_reg_score"] = profile_df[
    ["Self_Control", "Satisfaction"]
].mean(axis=1)

In [28]:
profile_df["self_reg_risk"] = -profile_df["self_reg_score"]

In [38]:
# content one-hot encoding
content_df = pd.get_dummies(
    profile_df[["Video_Category", "Watch_Reason"]],
    prefix=["Video_Category", "Watch_Reason"],
    drop_first=False
)

content_df = content_df.astype(int)

In [39]:
print(content_df.head())
print(content_df.columns.tolist())

   Video_Category_ASMR  Video_Category_Comedy  Video_Category_Entertainment  \
0                    0                      0                             0   
1                    0                      0                             0   
2                    0                      0                             0   
3                    0                      0                             0   
4                    0                      0                             0   

   Video_Category_Gaming  Video_Category_Jokes/Memes  \
0                      0                           0   
1                      0                           0   
2                      0                           0   
3                      0                           0   
4                      1                           0   

   Video_Category_Life Hacks  Video_Category_Pranks  Video_Category_Trends  \
0                          0                      1                      0   
1                          0    

In [40]:
# final dataset
final_df = pd.concat([
    profile_df[[
        "usage_score",
        "interaction_score",
        "self_reg_risk",
        "Scroll_Rate",
        "Frequency",
        "Self_Control",
        "Addiction_Level",
        "ProductivityLoss"
    ]],
    content_df
], axis=1)

In [41]:
# rearrange
content_cols = content_df.columns.tolist()

final_df = final_df[
    ["usage_score", "interaction_score", "self_reg_risk",
     "Scroll_Rate", "Self_Control"]
    + content_cols
    + ["Addiction_Level", "ProductivityLoss"]
]

In [42]:
print(final_df.columns.tolist())
print(final_df.head())

['usage_score', 'interaction_score', 'self_reg_risk', 'Scroll_Rate', 'Self_Control', 'Video_Category_ASMR', 'Video_Category_Comedy', 'Video_Category_Entertainment', 'Video_Category_Gaming', 'Video_Category_Jokes/Memes', 'Video_Category_Life Hacks', 'Video_Category_Pranks', 'Video_Category_Trends', 'Video_Category_Vlogs', 'Watch_Reason_Boredom', 'Watch_Reason_Entertainment', 'Watch_Reason_Habit', 'Watch_Reason_Procrastination', 'Addiction_Level', 'ProductivityLoss']
   usage_score  interaction_score  self_reg_risk  Scroll_Rate  Self_Control  \
0     0.072221           1.131136       0.005391     1.275597     -1.017757   
1     0.688139           0.098105      -0.009214    -0.129321     -0.045687   
2    -1.164820           1.286996      -0.016516     1.309863      0.440348   
3     0.731280           0.307372       0.005391     1.481195     -1.017757   
4    -0.410930          -1.111585      -0.031120    -1.568505      1.412417   

   Video_Category_ASMR  Video_Category_Comedy  Video_Ca

In [43]:
# sanity check
check = final_df.groupby("Addiction_Level")[[
    "usage_score", "interaction_score", "self_reg_risk"
]].mean()

print(check)

                 usage_score  interaction_score  self_reg_risk
Addiction_Level                                               
0                   0.105889          -0.003760       0.017332
1                  -0.068212          -0.030068      -0.023818
2                  -0.057975           0.010377      -0.014615
3                  -0.003311          -0.089775      -0.004766
4                  -0.038353           0.147864      -0.001911
5                   0.021106           0.001407       0.005391
6                  -0.146179           0.229852       0.012693
7                   0.026832           0.013140       0.019995


In [44]:
# sanity check
check = final_df.groupby("Addiction_Level")[[
    "usage_score", "interaction_score", "self_reg_risk"
]].median()

print(check)

                 usage_score  interaction_score  self_reg_risk
Addiction_Level                                               
0                   0.090177          -0.012773      -0.031120
1                  -0.051069           0.019143      -0.023818
2                  -0.061093           0.066991      -0.016516
3                   0.025597          -0.163304      -0.009214
4                  -0.017434           0.145407      -0.001911
5                   0.052559          -0.012544       0.005391
6                  -0.155674           0.207494       0.012693
7                   0.032360          -0.008418       0.019995


In [45]:
# high-risk
threshold = final_df["Addiction_Level"].median()

final_df["risk_binary"] = (final_df["Addiction_Level"] >= threshold).astype(int)

# mean difference
check_binary = final_df.groupby("risk_binary")[[
    "usage_score",
    "interaction_score",
    "self_reg_risk",
    "Scroll_Rate",
    "Self_Control"
]].mean()

print(check_binary)

             usage_score  interaction_score  self_reg_risk  Scroll_Rate  \
risk_binary                                                               
0               0.001208           0.000190      -0.003963     0.001003   
1              -0.001151          -0.000181       0.003777    -0.000956   

             Self_Control  
risk_binary                
0                0.858656  
1               -0.818407  


In [46]:
content_cols = [c for c in final_df.columns if "Watch_Reason_" in c]

print(final_df.groupby("risk_binary")[content_cols].mean().T.sort_values(by=1, ascending=False).head(10))

risk_binary                          0         1
Watch_Reason_Habit            0.319672  0.357422
Watch_Reason_Entertainment    0.264344  0.263672
Watch_Reason_Boredom          0.311475  0.244141
Watch_Reason_Procrastination  0.104508  0.134766


In [47]:
final_df.to_csv("behavior_profile_dataset.csv", index=False)